In [0]:
# %sql

# -- Create a replica of target table to test above migration script
# CREATE TABLE default.factrequestdetails_test
# DEEP CLONE default.factrequestdetails;

# -- CREATE TABLE default.factrequestdetails_backup
# -- DEEP CLONE default.factrequestdetails;

# -- DROP TABLE IF EXISTS default.factrequestdetails_test;
# -- DROP TABLE IF EXISTS default.factrequestdetails_backup;


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, BooleanType
from datetime import datetime

dbutils.widgets.text("bucket", "DataStagingBucket", "S3 Bucket")
dbutils.widgets.text("env", "prod", "Environment")

# 1. Define the path to CSV file
s3_bucket = dbutils.widgets.get("bucket")
env = dbutils.widgets.get("env")

# -------------------- Edit these for csv path, target table name and schema -------------------

file_path = f"s3a://{s3_bucket}/edw_exports/{env}/factRequestDetails__202603061440.csv"

# target_table_name = "default.factrequestdetails_test"
target_table_name = "default.factrequestdetails"

# schema mapping CSV's columns
csv_schema = StructType([
    StructField("runcycleid", IntegerType(), True),
    StructField("foirequestid", IntegerType(), True),
    StructField("requesttypeid", IntegerType(), True),
    StructField("requeststatusid", IntegerType(), True),
    StructField("receivedmodeid", IntegerType(), True),
    StructField("deliverymodeid", IntegerType(), True),
    StructField("applicantcategoryid", IntegerType(), True),
    StructField("requesteddate", TimestampType(), True),
    StructField("receiveddate", TimestampType(), True),
    StructField("startdate", TimestampType(), True),
    StructField("createddate", TimestampType(), True),
    StructField("modifieddate", TimestampType(), True),
    StructField("assigneddate", TimestampType(), True),
    StructField("receivedbyusername", StringType(), True),
    StructField("modifiedbyusername", StringType(), True),
    StructField("assignedbyid", IntegerType(), True),
    StructField("assignedtoid", IntegerType(), True),
    StructField("primaryusername", StringType(), True),
    StructField("primarygroupid", IntegerType(), True),
    StructField("groupname", StringType(), True),
    StructField("officeid", StringType(), True),
    StructField("ministry", StringType(), True),
    StructField("subject", StringType(), True),
    StructField("screeneddate", TimestampType(), True),
    StructField("targetdate", TimestampType(), True),
    StructField("amendment", StringType(), True),
    StructField("amendmentcreateddate", TimestampType(), True),
    StructField("amendmentcreatedby", StringType(), True),
    StructField("completeddate", TimestampType(), True),
    StructField("completedby", StringType(), True),
    StructField("deliverydate", TimestampType(), True),
    StructField("deliveredby", StringType(), True),
    StructField("closeddate", TimestampType(), True),
    StructField("closedby", StringType(), True),
    StructField("notes", StringType(), True),
    StructField("withheldstage", StringType(), True),
    StructField("otheraddress", StringType(), True),
    StructField("holdstage", StringType(), True),
    StructField("holddate", TimestampType(), True),
    StructField("appealtype", StringType(), True),
    StructField("reviewid", StringType(), True),
    StructField("withhelddate", TimestampType(), True),
    StructField("disposition", StringType(), True),
    StructField("execcomments", StringType(), True),
    StructField("originaltargetdate", TimestampType(), True),
    StructField("redactiondescription", StringType(), True),
    StructField("currentactivity", StringType(), True),
    StructField("onholddays", IntegerType(), True),
    StructField("processeddays", IntegerType(), True),
    StructField("releaseformat", StringType(), True),
    StructField("denialauthority", StringType(), True),
    StructField("caseowner", StringType(), True),
    StructField("caseownertitle", StringType(), True),
    StructField("caseowneremail", StringType(), True),
    StructField("caseownerphone", StringType(), True),
    StructField("originalcloseddate", TimestampType(), True),
    StructField("stdduedate", TimestampType(), True),
    StructField("remainingdays", IntegerType(), True),
    StructField("requestage", IntegerType(), True),
    StructField("currentactivitydate", TimestampType(), True),
    StructField("crossgovtno", StringType(), True),
    StructField("requestdescription", StringType(), True),
    StructField("requeststatus", StringType(), True),
    StructField("status", StringType(), True),
    StructField("countontime", IntegerType(), True),
    StructField("countoverdue", IntegerType(), True),
    StructField("daysoverdue", IntegerType(), True),
    StructField("publication", StringType(), True),
    StructField("publicationreason", StringType(), True),
    StructField("oipcno", StringType(), True),
    StructField("judicialreview", StringType(), True),
    StructField("reviewtype", StringType(), True),
    StructField("reason", StringType(), True),
    StructField("portfolioofficer", StringType(), True),
    StructField("passduedays", IntegerType(), True),
    StructField("description", StringType(), True),
    StructField("priority", StringType(), True),
    StructField("requesterid", IntegerType(), True),
    StructField("onbehalfofrequesterid", IntegerType(), True),
    StructField("referencenumber", StringType(), True),
    StructField("refvisualrequestfilenumber", StringType(), True),
    StructField("extension", StringType(), True),
    StructField("shipaddressid", IntegerType(), True),
    StructField("billaddressid", IntegerType(), True),
    StructField("originalreceiveddate", TimestampType(), True),
    StructField("coordinatednrresponsereqd", BooleanType(), True), 
    StructField("applicantfilereference", StringType(), True),
    StructField("secondaryusers", StringType(), True),
    StructField("noofdocdelivered", IntegerType(), True),
    StructField("activeflag", StringType(), True), 
    StructField("customfieldstatus", StringType(), True),
    StructField("visualrequestfilenumber", StringType(), True)
])

# 2. Read the CSV into a DataFrame
csv_df = (spark.read
    .format("csv")
    .option("header", "true")
    .schema(csv_schema)
    .load(file_path)
)

# 3. Add the hardcoded column to the source DataFrame
csv_df_updated = csv_df.withColumns({
    "sourceoftruth": lit("AXIS"),
    "assignedby": lit(None).cast(StringType()),
    "assignedto": lit(None).cast(StringType()),
    "onholdotherdays": lit(None).cast(IntegerType()),
    "isconsultflag": lit(None).cast(StringType()),
    "isphasedrelease": lit(None).cast(StringType()),
    "isoipcreview": lit(None).cast(StringType()),
    "isiaorestricted": lit(None).cast(StringType()),
    "isministryrestricted": lit(None).cast(StringType())
})

print(f"Number of rows in CSV DataFrame: {csv_df_updated.count()}")

# 4. Upsert (Merge) into the target Delta table
if spark.catalog.tableExists(target_table_name):
    delta_table = DeltaTable.forName(spark, target_table_name)
    
    # merge condition
    merge_condition = """
        target.foirequestid = source.foirequestid AND 
        target.runcycleid = source.runcycleid AND
        target.sourceoftruth = source.sourceoftruth
    """
    
    (delta_table.alias("target")
        .merge(
            csv_df_updated.alias("source"),
            merge_condition 
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Migration complete: CSV data upserted.")

else:
    (csv_df_updated.write
        .format("delta")
        .mode("overwrite") 
        .saveAsTable(target_table_name)
    )
    print("Migration complete: New Delta table created from CSV.")